# wrap-forward-fn-generic — faded example 3: Conditional boxing of the output

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `wrap-forward-fn-generic`. The last cell reports your progress on the `Backprop: wrap forward fn` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: wrap forward fn` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`wrap-forward-fn-generic`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "wrap-forward-fn-generic"
DD_SUBTOPIC = "Backprop: wrap forward fn"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Some ops return non-arrays (an int, a `torch.Size`). The wrapper must box the result only when it is a `torch.Tensor`; otherwise it returns the raw Python value so int-returning ops keep working.

## Faded exercise 3

### Box only tensor outputs

Implement `wrap_forward_fn(fwd_fn)` so the result is wrapped in `Tensor` only if it is a `torch.Tensor`, and returned unchanged otherwise. Complete the conditional that decides whether to box.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch as t

class Tensor:
    def __init__(self, array):
        self.array = array

def wrap_forward_fn(fwd_fn):
    def tensor_func(*args, **kwargs):
        raw_args = [a.array if isinstance(a, Tensor) else a for a in args]
        result = fwd_fn(*raw_args, **kwargs)
        if isinstance(result, t.Tensor):
            return Tensor(result)
        return result
    return tensor_func

f = wrap_forward_fn(lambda x: int(x.numel()))
print(f(Tensor(t.zeros(3, 4))))


def _test():
    # int-returning op -> bare int, NOT boxed
    numel = wrap_forward_fn(lambda x: int(x.numel()))
    n = numel(Tensor(t.zeros(3, 4)))
    assert isinstance(n, int) and n == 12
    # tensor-returning op -> boxed Tensor
    relu = wrap_forward_fn(lambda x: x.clamp(min=0))
    out = relu(Tensor(t.tensor([-1.0, 2.0])))
    assert isinstance(out, Tensor)
    assert t.allclose(out.array, t.tensor([0.0, 2.0]))


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

class Tensor:
    def __init__(self, array):
        self.array = array

def wrap_forward_fn(fwd_fn):
    def tensor_func(*args, **kwargs):
        raw_args = [a.array if isinstance(a, Tensor) else a for a in args]
        result = fwd_fn(*raw_args, **kwargs)
        if isinstance(result, t.Tensor):
            return Tensor(result)
        return result
    return tensor_func

f = wrap_forward_fn(lambda x: int(x.numel()))
print(f(Tensor(t.zeros(3, 4))))
```
</details>